In [2]:
import numpy as np
import pandas as pd


In [3]:
df=pd.read_csv('student_data.csv')

In [4]:
df.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 33 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   school      395 non-null    str  
 1   sex         395 non-null    str  
 2   age         395 non-null    int64
 3   address     395 non-null    str  
 4   famsize     395 non-null    str  
 5   Pstatus     395 non-null    str  
 6   Medu        395 non-null    int64
 7   Fedu        395 non-null    int64
 8   Mjob        395 non-null    str  
 9   Fjob        395 non-null    str  
 10  reason      395 non-null    str  
 11  guardian    395 non-null    str  
 12  traveltime  395 non-null    int64
 13  studytime   395 non-null    int64
 14  failures    395 non-null    int64
 15  schoolsup   395 non-null    str  
 16  famsup      395 non-null    str  
 17  paid        395 non-null    str  
 18  activities  395 non-null    str  
 19  nursery     395 non-null    str  
 20  higher      395 non-null    str  
 21  inte

In [6]:
df.isnull()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
390,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
391,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
392,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
393,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [7]:
df.isnull().sum().sum()

np.int64(0)

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor


X = df.drop(columns=["G1", "G2", "G3"])
y = df["G3"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric_features = X.select_dtypes(include="number").columns
categorical_features = X.select_dtypes(exclude="number").columns

numeric_scaled = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

numeric_unscaled = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_lr = ColumnTransformer([
    ("num", numeric_scaled, numeric_features),
    ("cat", categorical, categorical_features)
])

preprocessor_rf = ColumnTransformer([
    ("num", numeric_unscaled, numeric_features),
    ("cat", categorical, categorical_features)
])

linear_model = Pipeline([
    ("preprocessor", preprocessor_lr),
    ("model", LinearRegression())
])

forest_model = Pipeline([
    ("preprocessor", preprocessor_rf),
    ("model", RandomForestRegressor(
        n_estimators=200, random_state=42
    ))
])

linear_model.fit(X_train, y_train)
forest_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](30,)","['school','sex','age',...,'Walc','health','absences']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,30
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated wi

In [9]:
linearreg_pred=linear_model.predict(X_test)

In [10]:
linearreg_pred

array([ 1.88508025,  7.45339224,  8.30868805, 11.55112413, 10.20284528,
        9.43314825, 13.21126069, 12.69040139, 10.26673682, 12.34152834,
       12.54156595,  9.55022368, 11.1237184 , 13.57028017, 10.59970023,
       11.09924413,  9.87547231,  9.13815931, 10.42408213,  9.43498675,
       10.58544049, 13.09788625, 11.89731253, 10.92955435, 10.94009722,
       13.87394054, 12.44925199, 14.10066044, 13.24230246, 14.34394987,
        9.53551425, 10.01885518, 13.34836313,  9.29379058,  9.80013267,
       10.40700807,  5.17093329, 10.52571416, 11.49633814,  6.61327209,
        3.28896612, 10.45090058, 10.93512562,  7.32945242, 10.75178117,
        9.60021083, 10.20712084,  8.09768894,  9.65558315, 13.65721043,
        9.34722702, 13.40292314, 14.52921344, 11.74522714,  7.80184621,
        7.86114808, 12.72502128,  8.92576179, 13.5913325 , 12.902532  ,
       14.59257935, 11.31940527,  0.01575681, 11.75146985,  6.5327078 ,
        9.08799421,  8.51835183, 10.62312783,  9.22109327, 15.21

In [11]:
RFreg_pred=forest_model.predict(X_test)


In [12]:
RFreg_pred

array([ 9.525,  9.415, 10.2  , 12.16 ,  9.98 ,  8.935, 14.655, 11.365,
       10.195, 12.09 , 12.805, 12.805, 11.545,  9.16 , 11.3  , 10.045,
        9.41 ,  9.965,  8.74 ,  7.58 , 12.88 , 11.195, 10.415,  9.37 ,
       10.495, 11.345, 11.73 , 10.92 , 11.705, 12.63 , 10.34 , 10.83 ,
       13.445, 11.62 , 10.23 ,  9.62 ,  1.345, 12.255, 12.49 , 10.07 ,
        8.155, 11.575, 11.825,  9.54 , 11.24 , 11.895, 10.46 ,  9.555,
       12.56 , 13.465,  9.615, 13.68 , 12.085, 11.505, 10.765, 10.985,
       11.865,  8.29 , 12.275, 12.63 , 13.425, 11.63 ,  9.145, 11.9  ,
       10.615, 10.085, 10.205, 10.43 , 10.865, 12.11 , 10.795, 11.075,
       12.66 , 14.825,  9.125, 11.76 , 12.865, 13.325,  8.675])

In [13]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [14]:

# Linear Regression
mae_lr = mean_absolute_error(y_test, linearreg_pred)
rmse_lr = np.sqrt(mean_squared_error(y_test, linearreg_pred))
r2_lr = r2_score(y_test, linearreg_pred)

# Random Forest
mae_rf = mean_absolute_error(y_test, RFreg_pred)
rmse_rf = np.sqrt(mean_squared_error(y_test, RFreg_pred))
r2_rf = r2_score(y_test, RFreg_pred)

# Side-by-side comparison
comparison = pd.DataFrame({
    'MAE': [mae_lr, mae_rf],
    'RMSE': [rmse_lr, rmse_rf],
    'R2': [r2_lr, r2_rf],
}, index=['Linear Regression', 'Random Forest']).round(3)

print(comparison)

                     MAE   RMSE     R2
Linear Regression  3.395  4.196  0.141
Random Forest      2.994  3.748  0.315


In [19]:
import os

print(os.listdir("/home/neev"))


project_path = os.path.join(
    "/home/neev",
    "Studentperformancepredictor_project"
)

print(project_path)
print(os.path.isdir(project_path))

['.bash_logout', '.bash_profile', '.bashrc', '.config', '.zshrc', '.local', '.cache', 'Desktop', 'Downloads', 'Templates', 'Public', 'Documents', 'Music', 'Pictures', 'Videos', 'Projects', '.gnupg', '.profile', '.var', '.themes', '.icons', '.android', '.wget-hsts', '.gphoto', '.anydesk', '.flutter', '.dart-tool', '.pub-cache', '.cargo', '.ipython', 'ai-ml', '.jupyter', 'scikit_learn_data', 'student-performance-linearRegression', '.gitconfig', '.ssh', 'logistic-regression-breast-cancer', 'KNNPractice', 'ames-housing-decision-tree', '.vscode-oss', '.vscode-oss-shared', '.copilot', '.vscode', '.vscode-shared', '.dotnet', 'fastapi-learning', '.python_history', 'Studentpreformancepredictor_project']
/home/neev/Studentperformancepredictor_project
False


In [15]:
import joblib

joblib.dump(
    forest_model,
    "/home/neev/Studentperformancepredictor_project/model.pkl"
)

FileNotFoundError: [Errno 2] No such file or directory: '/home/neev/Studentperformancepredictor_project/model.pkl'

In [ ]:

import pickle 
pickle_model_path="model.pkl"
with open(pickle_model_path,"wb") as f:
    pickle.dump(Pipeline,f)